<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/notebook_ipynb/big_C_cloudflare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neccessary Run

In [1]:
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""
    <style>
        #scroll_output {
            height: 50px; /* Approximately 2 lines */
            overflow-y: scroll;
            background-color: #1e1e1e;
            color: #00ff00;
            padding: 10px;
            font-family: monospace;
            font-size: 14px;
            border: 1px solid #444;
            display: flex;
            flex-direction: column;
        }
    </style>
    <div id="scroll_output">Starting installation...</div>
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            # Escape single quotes for JS and update the div
            escaped_line = line.replace("'", "\\'").strip()
            display(HTML(f"""<script>
                var obj = document.getElementById('scroll_output');
                obj.innerHTML += '<div>' + '{escaped_line}' + '</div>';
                obj.scrollTop = obj.scrollHeight;
            </script>"""), display_id='scroll_update')
        process.wait()

# 2. Updated list of commands (Consolidated and Optimized)
commands_to_run = [
    # 1. Download and install the official Google Chrome stable version
    # "wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -",
    # "sh -c 'echo \"deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main\" >> /etc/apt/sources.list.d/google-chrome.list'",
    "apt-get -y update",
    "apt-get install -y google-chrome-stable",

    # 2. Install all Python dependencies in one step (quiet mode)
    "pip install selenium beautifulsoup4 pandas polars playwright chromedriver-autoinstaller xlsxwriter fastexcel curl_cffi scrapling patchright msgspec browserforge nest_asyncio easyocr -q",

    # 3. Install Playwright and Patchright browsers and their OS dependencies
    "playwright install chromium",
    "playwright install-deps",
    "patchright install chromium",
    "patchright install-deps"
]

run_and_scroll(commands_to_run)
print("\n✅ All installations finished.")


✅ All installations finished.


In [ ]:
pip install apify-fingerprint-datapoints -U

In [2]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-08-07


In [3]:
# @title Processing OCR
import io
import re
import requests
import easyocr
import polars as pl
import numpy as np
from PIL import Image

# 1. Initialize OCR
print("🔄 Initializing OCR Engine...")
reader = easyocr.Reader(['en', 'th'])

def get_condition_from_text(raw_text):
    """
    Handles dynamic patterns: Buy 2/3/4/5 Get 1 and Buy 2/3/4 Cheaper.
    Also normalizes Thai keywords and common OCR typos.
    """
    # Normalize text
    t = raw_text.upper().replace(" ", "")
    t = t.replace("BUV", "BUY")  # Fix common OCR typo

    # Extract the first digit found in the text (e.g., '3' from 'BUY3GET1')
    digits = re.findall(r'\d', t)
    n = digits[0] if digits else ""

    # --- 1. Supersave Case ---
    if any(k in t for k in ["SUPERSAVE", "SAVE", "ประหยัด"]):
        return "Supersave"

    # --- 2. Buy N Get 1 Case (Thai & English) ---
    if any(k in t for k in ["GET", "แถม"]):
        if n:
            # Special case for 1 Get 1
            if n == "1" or "1แถม1" in t or "1GET1" in t:
                return "Buy 1 Get 1"
            return f"Buy {n} Get 1"
        return "Buy 1 Get"

    # --- 3. Buy N Cheaper Case ---
    if "CHEAPER" in t:
        if n:
            return f"Buy {n} Cheaper"
        return "Buy 2 Cheaper"

    # Fallback: Return raw text if no pattern matches
    return raw_text.strip() if raw_text.strip() else None



🔄 Initializing OCR Engine...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [4]:
# @title udf data-prep
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )



In [5]:
# @title udf Transform (Fixed Volume Extraction)
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    Fixed to handle thousands separators (e.g., 1,300 ml).
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    # Added [\d,.]+ to capture digits, commas, and dots
    # Updated pattern:
    # 1. Added (?)i for case-insensitivity
    # 2. Removed \b to ensure Thai characters don't get blocked by boundary logic
    quant_unit_pattern = r"(?i)([\d.]+)\s*(ML|G|KG|L|GRAMS?)"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Fixed Volume Extraction:
        # 1. Extract the group (e.g., "1,300")
        # 2. Replace commas with nothing so it becomes "1300"
        # 3. Cast to Integer
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Int64, strict=False)
          .alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

# Scrape entire catalog in url list

In [6]:
%%script echo skipping
# @title work
# This code works
import asyncio
import nest_asyncio
import pandas as pd
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages():
    base_url = "https://www.bigc.co.th/en/category/laundry?brand=184%2C249%2C188%2C189%2C185"
    current_page = 2
    all_data = []

    # กำหนดค่าภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้า...")

    while True:
        target_url = f"{base_url}&page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True,
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # ถ้าไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A"
                    })

                # เพิ่มเลขหน้าเพื่อไปหน้าถัดไป
                current_page += 1

                # ใส่หน่วงเวลาเล็กน้อยเพื่อไม่ให้โดนบล็อก
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์
    if all_data:
        df = pd.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 2 ถึง {current_page-1}")
        display(df)
        df.to_csv('bigc_laundry_full.csv', index=False)
        print("บันทึกข้อมูลลงไฟล์ bigc_laundry_full.csv เรียบร้อยแล้ว")
    else:
        print("ไม่พบข้อมูลที่จะบันทึก")

# เริ่มทำงาน
await scrape_bigc_multi_pages()

skipping


In [7]:
# @title UDF scrape big C polars
# 1. ติดตั้งไลบรารีที่จำเป็น (รวมถึง polars)
# !pip install scrapling patchright msgspec browserforge nest_asyncio polars -q
# !patchright install chromium
# !patchright install-deps

import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def get_scrape_bigc_multi_pages_polars(base_url):
    current_page = 1
    all_data = []

    # กำหนดค่าคุกกี้และ headers เพื่อรักษาเซสชันภาษาอังกฤษ
    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print("เริ่มการดึงข้อมูลหลายหน้าด้วย Polars...")

    while True:
        # ตรวจสอบโครงสร้าง URL เพื่อต่อพารามิเตอร์ page ให้ถูกต้อง
        separator = "&" if "?" in base_url else "?"
        target_url = f"{base_url}{separator}page={current_page}"
        print(f"กำลังดึงข้อมูลหน้า {current_page}: {target_url}")

        try:
            # ใช้ StealthyFetcher เพื่อข้าม Cloudflare WAF
            page = await StealthyFetcher.async_fetch(
                target_url,
                headless=True,
                solve_cloudflare=True, # จัดการกับระบบตรวจสอบบอทอัตโนมัติ
                cookies=en_cookies,
                headers=en_headers,
                timeout=60000,
                network_idle=True
            )

            if page.status == 200:
                containers = page.css('div[class*="productCard_container"]')

                # หากไม่พบสินค้าในหน้านี้ แสดงว่าถึงหน้าสุดท้ายแล้ว
                if not containers:
                    print(f"ไม่พบสินค้าเพิ่มเติมที่หน้า {current_page} จบการทำงาน")
                    break

                print(f"พบสินค้า {len(containers)} รายการในหน้า {current_page}")

                for item in containers:
                    # สกัดข้อมูลชื่อสินค้าและราคาจาก DOM
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()

                    # # Extract the availability status/badge
                    # status = item.css('div[class*="productCard_badge"]::text').get()
                    # if not status:
                    #     status = item.css('div[class*="productCard_label"]::text').get()

                    # # Map "Available soon" or "Out of stock"
                    # is_available = True
                    # if status and ("Soon" in status or "หมด" in status):
                    #     is_available = False

                    # Check for the "Available soon" or "Sold out" badge
                    # Usually found in productCard_badge or productCard_label
                    badge_text = item.css('div[class*="productCard_badge"]::text').get() or ""

                    all_data.append({
                        "page": current_page,
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A",
                        "status": badge_text.strip() if badge_text else "Available"
                    })

                current_page += 1
                # หน่วงเวลาเล็กน้อยเพื่อหลีกเลี่ยงการถูกตรวจจับพฤติกรรม
                await asyncio.sleep(2)
            else:
                print(f"เกิดข้อผิดพลาดที่หน้า {current_page} (Status: {page.status}) หยุดการทำงาน")
                break

        except Exception as e:
            print(f"เกิดข้อผิดพลาดที่หน้า {current_page}: {e}")
            break

    # สรุปผลลัพธ์และส่งคืนค่าเป็น Polars DataFrame
    if all_data:
        df = pl.DataFrame(all_data)
        print(f"\nดึงข้อมูลสำเร็จทั้งหมด {len(df)} รายการ จากหน้า 1 ถึง {current_page-1}")
        return df
    else:
        print("ไม่พบข้อมูลที่จะประมวลผล")
        return pl.DataFrame()

In [8]:
import asyncio
import nest_asyncio
import polars as pl
from scrapling.fetchers import StealthyFetcher

nest_asyncio.apply()

async def scrape_bigc_multi_pages(base_url_list: list):
    all_data = []

    en_cookies = [
        {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
        {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
    ]
    en_headers = {'Accept-Language': 'en-US,en;q=0.9'}

    print(f"🚀 Starting Multi-Category Scrape ({len(base_url_list)} categories)...")

    for base_url in base_url_list:
        current_page = 1
        print(f"\n📂 Processing Category: {base_url}")

        while True:
            separator = "&" if "?" in base_url else "?"
            # Ensure we don't stack multiple page params
            clean_url = base_url.split(f"{separator}page=")[0]
            target_url = f"{clean_url}{separator}page={current_page}"

            print(f"   📄 Page {current_page}...")

            try:
                page = await StealthyFetcher.async_fetch(
                    target_url,
                    headless=True,
                    solve_cloudflare=True,
                    cookies=en_cookies,
                    headers=en_headers,
                    timeout=60000,
                    network_idle=True
                )

                if page.status != 200:
                    print(f"   🛑 Stopped at page {current_page} (Status: {page.status})")
                    break

                containers = page.css('div[class*="productCard_container"]')
                if not containers:
                    print(f"   ✅ Category Complete.")
                    break

                for item in containers:
                    name = item.css('div[class*="productCard_title"] a::text').get()
                    sale_price = item.css('span[class*="productCard_sale_price"]::text').get()
                    original_price = item.css('div[class*="productCard_base_price"]::text').get()
                    badge_url = item.css('div[class*="productCard_badge"] img::attr(src)').get()

                    all_data.append({
                        "product_name": name.strip() if name else "N/A",
                        "sale_price": sale_price.strip() if sale_price else "N/A",
                        "original_price": original_price.strip().replace('฿', '') if original_price else "N/A",
                        "condition": None, # Placeholder for Part 2
                        "badge_url": badge_url if badge_url else "null"
                    })

                current_page += 1
                await asyncio.sleep(1) # Be polite to the server

            except Exception as e:
                print(f"   ❌ Error at page {current_page}: {e}")
                break

    return pl.DataFrame(all_data) if all_data else pl.DataFrame()

# --- RUN SCRAPER ---
category_list = [
    "https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100",
    "https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100"
]

final_df = await scrape_bigc_multi_pages(category_list)
print(f"🏁 Scrape finished. Total products: {len(final_df)}")

🚀 Starting Multi-Category Scrape (2 categories)...

📂 Processing Category: https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100
   📄 Page 1...


[2026-08-07 07:23:10] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:23:27] INFO: Cloudflare page didn't disappear after 10s, continuing...
INFO:scrapling:Cloudflare page didn't disappear after 10s, continuing...
[2026-08-07 07:23:27] INFO: Looks like Cloudflare captcha is still present, solving again
INFO:scrapling:Looks like Cloudflare captcha is still present, solving again
[2026-08-07 07:23:27] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:23:37] INFO: Cloudflare captcha is solved
INFO:scrapling:Cloudflare captcha is solved
[2026-08-07 07:24:37] INFO: Fetched (307) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (307) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=

   📄 Page 2...


[2026-08-07 07:24:41] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:24:54] INFO: Cloudflare page didn't disappear after 10s, continuing...
INFO:scrapling:Cloudflare page didn't disappear after 10s, continuing...
[2026-08-07 07:24:54] INFO: Looks like Cloudflare captcha is still present, solving again
INFO:scrapling:Looks like Cloudflare captcha is still present, solving again
[2026-08-07 07:24:54] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:25:03] INFO: Cloudflare captcha is solved
INFO:scrapling:Cloudflare captcha is solved
[2026-08-07 07:26:04] INFO: Fetched (307) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (307) <GET https://www.bigc.co.th/en/category/laundry?brand=187%2C188%2C249%2C186%2C256&limit=100&page=

   📄 Page 3...


...

   📄 Page 2...


[2026-08-07 07:31:49] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:32:01] INFO: Cloudflare page didn't disappear after 10s, continuing...
INFO:scrapling:Cloudflare page didn't disappear after 10s, continuing...
[2026-08-07 07:32:01] INFO: Looks like Cloudflare captcha is still present, solving again
INFO:scrapling:Looks like Cloudflare captcha is still present, solving again
[2026-08-07 07:32:01] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:32:08] INFO: Cloudflare captcha is solved
INFO:scrapling:Cloudflare captcha is solved
[2026-08-07 07:33:08] INFO: Fetched (307) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (307) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=2> (referer: https://www.google.com/)
[2026-08

   📄 Page 3...


[2026-08-07 07:33:13] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:33:26] INFO: Cloudflare page didn't disappear after 10s, continuing...
INFO:scrapling:Cloudflare page didn't disappear after 10s, continuing...
[2026-08-07 07:33:26] INFO: Looks like Cloudflare captcha is still present, solving again
INFO:scrapling:Looks like Cloudflare captcha is still present, solving again
[2026-08-07 07:33:26] INFO: The turnstile version discovered is "managed"
INFO:scrapling:The turnstile version discovered is "managed"
[2026-08-07 07:33:32] INFO: Cloudflare captcha is solved
INFO:scrapling:Cloudflare captcha is solved
[2026-08-07 07:34:32] INFO: Fetched (307) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3> (referer: https://www.google.com/)
INFO:scrapling:Fetched (307) <GET https://www.bigc.co.th/en/category/dishwashing-liquid?limit=100&page=3> (referer: https://www.google.com/)
[2026-08

   ✅ Category Complete.
🏁 Scrape finished. Total products: 412


In [9]:
# --- START PROCESSING ---

# 2. Get unique badge URLs (Excluding nulls)
unique_urls = [url for url in final_df["badge_url"].unique().to_list() if url and url != "null"]

if not unique_urls:
    print("⚠️ No valid badge URLs found.")
else:
    print(f"🔍 Processing {len(unique_urls)} unique badges...")
    badge_map = {}

    for url in unique_urls:
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:
                img = Image.open(io.BytesIO(response.content)).convert('RGB')

                # Upscale for OCR accuracy
                img = img.resize((img.width * 4, img.height * 4), resample=Image.LANCZOS)

                # Convert to Numpy Array for EasyOCR
                img_array = np.array(img)

                # Run OCR
                results = reader.readtext(img_array)
                raw_text = " ".join([res[1] for res in results])

                # Process text through our new dynamic function
                label = get_condition_from_text(raw_text)
                badge_map[url] = label
                print(f"✅ {url[-15:]} -> {label}")
            else:
                badge_map[url] = None
        except Exception as e:
            print(f"❌ Error on {url[-15:]}: {e}")
            badge_map[url] = None

    # 3. Apply the results back to the 'condition' column
    final_df = final_df.with_columns(
        pl.col("badge_url").replace(badge_map).alias("condition")
    )

    print("\n✨ Process Complete!")
    # Show results where a condition was found
    print(final_df.filter(pl.col("condition").is_not_null()).head(20))

🔍 Processing 4 unique badges...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ 56180476459.png -> Buy 2 Cheaper


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ 56180890355.png -> Buy 1 Get 1
✅ 67563483039.png -> Supersave
✅ 56180931816.png -> Buy 2 Get 1

✨ Process Complete!
shape: (20, 5)
┌────────────────────────────┬────────────┬────────────────┬───────────┬───────────────────────────┐
│ product_name               ┆ sale_price ┆ original_price ┆ condition ┆ badge_url                 │
│ ---                        ┆ ---        ┆ ---            ┆ ---       ┆ ---                       │
│ str                        ┆ str        ┆ str            ┆ str       ┆ str                       │
╞════════════════════════════╪════════════╪════════════════╪═══════════╪═══════════════════════════╡
│ HYGIENE Expert Care        ┆ 47.00      ┆ 60.00          ┆ Supersave ┆ https://st.bigc-cs.com/cd │
│ Concentrat…                ┆            ┆                ┆           ┆ n-cgi…                    │
│ FINELINE Fabric Softener   ┆ 85.00      ┆ 89.00          ┆ Supersave ┆ https://st.bigc-cs.com/cd │
│ Sunsh…                     ┆            ┆                

In [10]:
final_df.to_pandas()

,product_name,sale_price,original_price,condition,badge_url
0,HYGIENE Expert Care Concentrated Fabric Soften...,47.00,60.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
1,"FINELINE Fabric Softener Sunshine Gold 1,300 ml.",85.00,89.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
2,FINELINE Plus Liquid Laundry Detergent Sunny G...,179.00,N/A,null,null
3,HYGIENE Fabric Softener Violet Soft Scent 500 ...,49.00,49.50,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
4,"FINELINE Fabric Softener Gentle White 1,300 ml.",85.00,89.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
...,...,...,...,...,...
407,BIG C HAPPY PRICE PRO Dishwashing Liquid Lemon...,73.00,87.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
408,BIG C HAPPY PRICE PRO Dishwashing Liquid Lemon...,118.00,139.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...
409,3M 2 in 1 Cleaner & Freshener Fresh Mint 500 ml.,385.00,N/A,null,null
410,SUNLIGHT Plus Mind & Care Dishwashing Liquid G...,29.00,34.00,Supersave,https://st.bigc-cs.com/cdn-cgi/image/format=we...


In [11]:
# final_df.write_excel(f"big_c_{today_date}.xlsx")

In [12]:
df_big_c = final_df.clone()

In [13]:
df_big_c_sel = df_big_c.select([
    pl.col("product_name").alias("name"),
    # strict=False จะเปลี่ยนทุกอย่างที่แปลงเป็น Float ไม่ได้ให้เป็น Null
    pl.col("sale_price").cast(pl.Float64, strict=False).alias("promotion_price"),
    pl.col("original_price").cast(pl.Float64, strict=False).alias("original_price"),
    pl.col("condition")
])

In [14]:
# How to use it:
df_prep_big_c = re_evaluate_price(df_big_c_sel)
df_trans_big_c = parse_product_names(df_prep_big_c, "BigC")

In [15]:
df_trans_big_c

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""HYGIENE Expert Care Concentrat…",47.0,60.0,"""Supersave""","""2026-08-07""","""HYGIENE""",480,"""ML""",null,"""BigC"""
"""FINELINE Fabric Softener Sunsh…",85.0,89.0,"""Supersave""","""2026-08-07""","""FINELINE""",300,"""ML""",null,"""BigC"""
"""FINELINE Plus Liquid Laundry D…",null,179.0,"""null""","""2026-08-07""","""FINELINE""",1250,"""ML""",null,"""BigC"""
"""HYGIENE Fabric Softener Violet…",49.0,49.5,"""Supersave""","""2026-08-07""","""HYGIENE""",500,"""ML""","""PACK 3""","""BigC"""
"""FINELINE Fabric Softener Gentl…",85.0,89.0,"""Supersave""","""2026-08-07""","""FINELINE""",300,"""ML""",null,"""BigC"""
…,…,…,…,…,…,…,…,…,…
"""BIG C HAPPY PRICE PRO Dishwash…",73.0,87.0,"""Supersave""","""2026-08-07""","""BIG""",2000,"""ML""",null,"""BigC"""
"""BIG C HAPPY PRICE PRO Dishwash…",118.0,139.0,"""Supersave""","""2026-08-07""","""BIG""",null,"""L""",null,"""BigC"""
"""3M 2 in 1 Cleaner & Freshener …",null,385.0,"""null""","""2026-08-07""","""3M""",500,"""ML""",null,"""BigC"""


In [16]:
df_trans_big_c.write_excel(f"big_c_result_{today_date}.xlsx")

# Find watchlist

In [17]:
# @title udf Scrape Watchlist (Patchright + Isolated Contexts)

import re
import random
import asyncio
import polars as pl
from bs4 import BeautifulSoup
from patchright.async_api import async_playwright

async def scrape_bigc_watchlist_unlimited(urls: list[str]) -> pl.DataFrame:
    """
    Unlimited Queue Scraper for Big C.
    Never drops a URL. Will keep pushing failed URLs to the back of the queue
    until every single item is successfully scraped.
    """
    extracted_data = []

    # Initialize the queue as a list of tuples: (url, current_attempt_number)
    queue = [(url, 1) for url in urls]

    print(f"Starting UNLIMITED scrape for {len(urls)} Big C products...")
    print("Strategy: Infinite Queue + Heavy Cooldowns for stubborn links.\n")

    while queue:
        # Pop the first item off the front of the queue
        current_url, attempt = queue.pop(0)
        print(f"Fetching: {current_url}")
        print(f"  -> (Attempt {attempt}) | Items remaining in queue: {len(queue) + 1}")

        success = False

        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            context = await browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
                viewport={"width": 1920, "height": 1080}
            )

            # 1. Update Domain to Big C
            await context.add_cookies([
                {'name': 'language', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'},
                {'name': 'NEXT_LOCALE', 'value': 'en', 'domain': '.bigc.co.th', 'path': '/'}
            ])

            page = await context.new_page()
            try:
                await page.goto(current_url, wait_until="domcontentloaded", timeout=60000)
                await asyncio.sleep(3) # Wait for page to settle or CF gate to appear

                # 2. Check for Cloudflare via body text
                body_locator = page.locator("body")
                body_text = await body_locator.inner_text()

                if "just a moment" in body_text.lower() or "blocked" in body_text.lower() or "cloudflare" in body_text.lower():
                    print("  -> Cloudflare challenge gate detected. Attempting active bypass...")

                    resolved = False
                    for tick in range(30):
                        await asyncio.sleep(1)
                        # Wiggle mouse to trigger bot sensors naturally
                        await page.mouse.move(random.randint(300, 800), random.randint(300, 800))

                        if tick % 5 == 0:
                            await page.mouse.click(960, 540) # Click Turnstile position

                        body_text = await body_locator.inner_text()
                        if "just a moment" not in body_text.lower() and "blocked" not in body_text.lower() and "cloudflare" not in body_text.lower():
                            print("  -> Cloudflare resolved!")
                            resolved = True
                            break

                    if not resolved:
                        print("  -> Still blocked after 30s. Triggering emergency reload...")
                        await page.reload(wait_until="domcontentloaded", timeout=60000)
                        await asyncio.sleep(5)
                        body_text = await body_locator.inner_text()

                        if "just a moment" not in body_text.lower() and "blocked" not in body_text.lower() and "cloudflare" not in body_text.lower():
                            resolved = True
                            print("  -> Cloudflare resolved after reload!")
                        else:
                            print("  -> Bypass failed on this turn.")
                else:
                    resolved = True

                # 3. Data Extraction phase
                if resolved:
                    await asyncio.sleep(2) # Let React/Next.js load the UI
                    html = await page.content()
                    soup = BeautifulSoup(html, "html.parser")

                    # Extract Name
                    name_tag = soup.find("h1")
                    name = name_tag.get_text(strip=True) if name_tag else None

                    promo = orig = condition = None

                    if name and "just a moment" not in name.lower():

                        # --- BIG C EXACT PRICE EXTRACTION ---
                        price_wrapper = soup.find("div", id="pdp_product-price-new")

                        if price_wrapper:
                            # Extract Original Price (Using its exact ID)
                            base_tag = price_wrapper.find("span", id="pdp_price-base")
                            if base_tag:
                                orig = base_tag.get_text(strip=True).replace(',', '')

                            # Extract Promo Price (Safely using regex to match the dynamic class)
                            promo_div = price_wrapper.find("div", class_=re.compile(r"productDetail_product_price_new"))
                            if promo_div:
                                direct_texts = [text for text in promo_div.find_all(string=True, recursive=False) if text.strip()]
                                if direct_texts:
                                    promo = direct_texts[0].strip().replace(',', '')

                        # Fallback just in case there is no promo and it's a standard priced item
                        if not promo and not orig:
                            fallback_tag = soup.find("span", class_=re.compile(r"productDetail_product_price_default"))
                            if fallback_tag:
                                orig = fallback_tag.get_text(strip=True).replace(',', '')

                        # --- BIG C BADGE EXTRACTION ---
                        badge_div = soup.find("div", class_=re.compile(r"imageSlider_badge_top-right"))
                        if badge_div:
                            img_tag = badge_div.find("img")
                            if img_tag and img_tag.get("src"):
                                badge_url = img_tag["src"]
                                if badge_url.startswith("/"):
                                    badge_url = "https://www.bigc.co.th" + badge_url
                                condition = badge_url

                        # Append to our data list
                        extracted_data.append({
                            "name": name,
                            "promotion_price": promo,
                            "original_price": orig,
                            "condition": condition,
                            "url": current_url
                        })

                        badge_status = "[Image Found]" if condition else "None"
                        print(f"  Successfully grabbed: {name[:35]}... | Orig: {orig} | Promo: {promo} | Badge: {badge_status}")
                        success = True
                    else:
                        print("  -> Extracted name is invalid. Likely still blocked.")

            except Exception as e:
                print(f"  Error: {str(e)[:80]}...")
            finally:
                if browser.is_connected():
                    await browser.close()

        # 4. UNLIMITED QUEUE LOGIC: Always re-queue on failure
        if not success:
            queue.append((current_url, attempt + 1))
            print(f"  [!] Failed. Pushing to the back of the queue (Will never drop).\n")

            if len(queue) == 1:
                print("  (Only 1 item left in queue. Taking a 30s heavy cooldown to reset IP flags...)")
                await asyncio.sleep(30)
            elif len(queue) <= 3:
                print("  (Queue is getting small. Taking a 15s cooldown...)")
                await asyncio.sleep(15)
            else:
                await asyncio.sleep(random.uniform(5, 8))
        else:
            wait_time = random.uniform(3, 5)
            if queue:
                print(f"  Waiting {wait_time:.1f}s before next item in queue...\n")
                await asyncio.sleep(wait_time)

    if not extracted_data:
        print("\nNo data collected.")
        return pl.DataFrame()

    df_raw = pl.DataFrame(extracted_data)

    return df_raw

In [18]:
# @title RUN Watchlist Scraper
watchlist_urls = [
# -- BIG C
# 'FINELINE Liquid Laundry Detergent Sunny Gold Scent 550 ml.',
"https://www.bigc.co.th/en/product/fineline-liquid-laundry-detergent-sunny-gold-scent-550-ml.3791984",
# 'FINELINE Plus Liquid Laundry Detergent Sunny Gold Scent 1250 ml.',
"https://www.bigc.co.th/en/product/fineline-plus-liquid-laundry-detergent-sunny-gold-scent-1250-ml.2155497",
# 'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 600 ml.',
"https://www.bigc.co.th/en/product/hygiene-expert-wash-concentrate-liquid-detergent-milky-touch-600-ml.6394917",
# 'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 1400 ml.',
"https://www.bigc.co.th/en/product/hygiene-expert-wash-concentrate-liquid-detergent-milky-touch-1400-ml.6394919",
# 'PAO Win Wash Liquid Laundry Detergent 620 ml.',
"https://www.bigc.co.th/en/product/pao-win-wash-liquid-detergent-620-ml.12782",
# 'PAO Win Wash Liquid Laundry Detergent 1300 ml.',
"https://www.bigc.co.th/en/product/pao-win-wash-concentrated-liquid-detergent-formula-1300-ml.34065",
# 'PAO Super White Laundry Detergent 1800 g.',
"https://www.bigc.co.th/en/product/pao-super-white-laundry-detergent-1800-g.5977",
# 'PAO Super White Powder Laundry Detergent 2400 g.',
"https://www.bigc.co.th/en/product/pao-super-white-detergent-2400-g.3520",
# 'ATTACK EASY DETERGENT HAPPY SWEET 2500 G',
"https://www.bigc.co.th/en/product/attack-easy-detergent-happy-sweet-2500-g.47463",
# 'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml.',
"https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-milky-touch-480-ml.12003",
# 'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 1000 ml. Pack 2',
"https://www.bigc.co.th/en/product/hygiene-expert-care-concentrated-fabric-softener-milky-touch-scent-1000-ml-pack-2.1953035",
# 'LIPON F Dishwashing Liquid Hygienic Formula 500 ml. Pack 3',
"https://www.bigc.co.th/en/product/lipon-f-dishwashing-liquid-hygienic-formula-500-ml-pack-3.3639",
# 'LIPON F Dishwashing Liquid Hygienic Formula 3200 ml.',
"https://www.bigc.co.th/en/product/lipon-f-dishwashing-liquid-hygienic-formula-3200-ml.673",
# 'PRO Blue Plus Powder Laundry Detergent 2400 g.',
"https://www.bigc.co.th/en/product/pro-blue-plus-powder-laundry-detergent-standard-formula-2400-g.501",
# 'LIPON F Sanitary Formula Dish Washing Liquid Refill 750 ml. Pack of 2',
"https://www.bigc.co.th/en/product/lipon-f-sanitary-formula-dish-washing-liquid-refill-750-ml-pack-of-2.78902",
# 'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml. Pack 2+1',
"https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-tender-touch-480-ml-pack-2-free-1.32428",
# 'ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1',
# NOT FOUND
# ATTACK EZ Conventional Detergent Happy Sweet Scent 1700 g.
"https://www.bigc.co.th/en/product/attack-ez-conventional-detergent-happy-sweet-scent-1700-g.47863",
]

# Run the sequential queue version
df_watchlist_final = await scrape_bigc_watchlist_unlimited(watchlist_urls)

print("\n--- Watchlist Scrape Complete ---")
display.display(df_watchlist_final)

Starting UNLIMITED scrape for 17 Big C products...
Strategy: Infinite Queue + Heavy Cooldowns for stubborn links.

Fetching: https://www.bigc.co.th/en/product/fineline-liquid-laundry-detergent-sunny-gold-scent-550-ml.3791984
  -> (Attempt 1) | Items remaining in queue: 17
  -> Cloudflare challenge gate detected. Attempting active bypass...
  -> Still blocked after 30s. Triggering emergency reload...
  -> Cloudflare resolved after reload!
  Successfully grabbed: FINELINE Liquid Laundry Detergent S... | Orig: None | Promo: 89.00 | Badge: None
  Waiting 3.5s before next item in queue...

Fetching: https://www.bigc.co.th/en/product/fineline-plus-liquid-laundry-detergent-sunny-gold-scent-1250-ml.2155497
  -> (Attempt 1) | Items remaining in queue: 16
  -> Cloudflare challenge gate detected. Attempting active bypass...
  -> Still blocked after 30s. Triggering emergency reload...
  -> Cloudflare resolved after reload!
  Successfully grabbed: FINELINE Plus Liquid Laundry Deterg... | Orig: None

name,promotion_price,original_price,condition,url
str,str,str,str,str
"""FINELINE Liquid Laundry Deterg…","""89.00""",null,null,"""https://www.bigc.co.th/en/prod…"
"""FINELINE Plus Liquid Laundry D…","""179.00""",null,null,"""https://www.bigc.co.th/en/prod…"
"""HYGIENE Expert Wash Concentrat…","""55.00""","""65.00""","""https://st.bigc-cs.com/cdn-cgi…","""https://www.bigc.co.th/en/prod…"
"""HYGIENE Expert Wash Concentrat…","""139.00""",null,null,"""https://www.bigc.co.th/en/prod…"
"""PAO Win Wash Liquid Laundry De…","""99.00""",null,"""https://st.bigc-cs.com/cdn-cgi…","""https://www.bigc.co.th/en/prod…"
…,…,…,…,…
"""LIPON F Dishwashing Liquid Hyg…","""165.00""",null,null,"""https://www.bigc.co.th/en/prod…"
"""PRO Blue Plus Powder Laundry D…","""150.00""",null,null,"""https://www.bigc.co.th/en/prod…"
"""LIPON F Sanitary Formula Dish …","""85.00""",null,null,"""https://www.bigc.co.th/en/prod…"


In [19]:
# @title Run OCR on Watchlist Badges
import io
import requests
import easyocr
import polars as pl
import numpy as np
from PIL import Image
import IPython.display as display

# 1. Initialize OCR (if not already done)
print("🔄 Initializing OCR Engine...")
reader = easyocr.Reader(['en', 'th'])

# 2. Get unique badge URLs from your dataframe (excluding null strings)
unique_urls = [
    url for url in df_watchlist_final["condition"].unique().to_list() # Changed from "badge_url" to "condition"
    if url and url != "null"
]

if not unique_urls:
    print("⚠️ No valid badge URLs found to process.")
else:
    print(f"🔍 Processing {len(unique_urls)} unique badges...")
    badge_map = {}

    for url in unique_urls:
        try:
            # Download the image
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code == 200:
                img = Image.open(io.BytesIO(response.content)).convert('RGB')

                # Upscale by 4x for better OCR accuracy on small badges
                img = img.resize((img.width * 4, img.height * 4), resample=Image.LANCZOS)
                img_array = np.array(img)

                # Read text
                results = reader.readtext(img_array)
                raw_text = " ".join([res[1] for res in results])

                # Route through your custom condition function!
                label = get_condition_from_text(raw_text)
                badge_map[url] = label
                print(f"✅ Extracted: '{raw_text}' -> Mapped to: {label}")
            else:
                badge_map[url] = None

        except Exception as e:
            print(f"❌ Error processing badge: {e}")
            badge_map[url] = None

    # Handle null mapping safely
    badge_map["null"] = None
    badge_map[None] = None

    # 3. Overwrite the condition column in df_watchlist_final
    df_watchlist_final = df_watchlist_final.with_columns(
        pl.col("condition").replace(badge_map, default=None).alias("condition") # Changed from "badge_url" to "condition"
    )

    print("\n✨ OCR Process Complete!")

    # Display the updated rows
    display.display(
        df_watchlist_final
        .select(["name", "promotion_price", "url", "condition"]) # Changed "badge_url" to "url"
        .filter(pl.col("condition").is_not_null())
    )

🔄 Initializing OCR Engine...
🔍 Processing 2 unique badges...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ Extracted: 'ถูกจริง ประหยัดจริง' -> Mapped to: Supersave
✅ Extracted: 'buv 1. get' -> Mapped to: Buy 1 Get 1

✨ OCR Process Complete!


/tmp/ipykernel_3366/2194567152.py:60: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("condition").replace(badge_map, default=None).alias("condition") # Changed from "badge_url" to "condition"


name,promotion_price,url,condition
str,str,str,str
"""HYGIENE Expert Wash Concentrat…","""55.00""","""https://www.bigc.co.th/en/prod…","""Supersave"""
"""PAO Win Wash Liquid Laundry De…","""99.00""","""https://www.bigc.co.th/en/prod…","""Buy 1 Get 1"""
"""HYGIENE Expert Care Concentrat…","""47.00""","""https://www.bigc.co.th/en/prod…","""Supersave"""
"""HYGIENE Expert Care Concentrat…","""195.00""","""https://www.bigc.co.th/en/prod…","""Supersave"""
"""LIPON F Dishwashing Liquid Hyg…","""69.00""","""https://www.bigc.co.th/en/prod…","""Supersave"""
"""HYGIENE Expert Care Concentrat…","""119.00""","""https://www.bigc.co.th/en/prod…","""Supersave"""
"""ATTACK EZ Conventional Deterge…","""89.00""","""https://www.bigc.co.th/en/prod…","""Supersave"""


In [20]:
df_watchlist_final.to_pandas()

,name,promotion_price,original_price,condition,url
0,FINELINE Liquid Laundry Detergent Sunny Gold S...,89.00,None,None,https://www.bigc.co.th/en/product/fineline-liq...
1,FINELINE Plus Liquid Laundry Detergent Sunny G...,179.00,None,None,https://www.bigc.co.th/en/product/fineline-plu...
2,HYGIENE Expert Wash Concentrate Liquid Deterge...,55.00,65.00,Supersave,https://www.bigc.co.th/en/product/hygiene-expe...
3,HYGIENE Expert Wash Concentrate Liquid Deterge...,139.00,None,None,https://www.bigc.co.th/en/product/hygiene-expe...
4,PAO Win Wash Liquid Laundry Detergent 620 ml.,99.00,None,Buy 1 Get 1,https://www.bigc.co.th/en/product/pao-win-wash...
5,PAO Win Wash Liquid Laundry Detergent 1300 ml.,185.00,None,None,https://www.bigc.co.th/en/product/pao-win-wash...
6,PAO Super White Laundry Detergent 1800 g.,112.00,None,None,https://www.bigc.co.th/en/product/pao-super-wh...
7,PAO Super White Powder Laundry Detergent 2400 g.,155.00,None,None,https://www.bigc.co.th/en/product/pao-super-wh...
8,ATTACK EASY DETERGENT HAPPY SWEET 2500 G,159.00,None,None,https://www.bigc.co.th/en/product/attack-easy-...
9,HYGIENE Expert Care Concentrated Fabric Soften...,47.00,60.00,Supersave,https://www.bigc.co.th/en/product/hygiene-fabr...


In [21]:
list_to_search = [
# -- BIG C
'FINELINE Liquid Laundry Detergent Sunny Gold Scent 550 ml.',
'FINELINE Plus Liquid Laundry Detergent Sunny Gold Scent 1250 ml.',
'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 600 ml.',
'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 1400 ml.',
'PAO Win Wash Liquid Laundry Detergent 620 ml.',
'PAO Win Wash Liquid Laundry Detergent 1300 ml.',
'PAO Super White Laundry Detergent 1800 g.',
'PAO Super White Powder Laundry Detergent 2400 g.',
'ATTACK EASY DETERGENT HAPPY SWEET 2500 G',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml.',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 1000 ml. Pack 2',
'LIPON F Dishwashing Liquid Hygienic Formula 500 ml. Pack 3',
'LIPON F Dishwashing Liquid Hygienic Formula 3200 ml.',
'PRO Blue Plus Powder Laundry Detergent 2400 g.',
'LIPON F Sanitary Formula Dish Washing Liquid Refill 750 ml. Pack of 2',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml. Pack 2+1',
'ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1',
# Found this, above (Not Found)
'ATTACK EZ Conventional Detergent Happy Sweet Scent 1700 g.'
]

search_df = pl.DataFrame({"product_name": list_to_search})

# Join with df_trans_lotuss to get original_price and promotion_price
search_results_df = search_df.join(
    df_watchlist_final.select(["name", "original_price", "promotion_price"]),
    left_on="product_name",
    right_on="name",
    how="left"
)
lotuss_names_set = set(df_watchlist_final["name"].to_list())

search_results_df = search_results_df.with_columns(
    pl.col("product_name").is_in(lotuss_names_set).alias("Found")
).unique()

print("Search Results with Prices:")
print(search_results_df)

Search Results with Prices:
shape: (18, 4)
┌─────────────────────────────────┬────────────────┬─────────────────┬───────┐
│ product_name                    ┆ original_price ┆ promotion_price ┆ Found │
│ ---                             ┆ ---            ┆ ---             ┆ ---   │
│ str                             ┆ str            ┆ str             ┆ bool  │
╞═════════════════════════════════╪════════════════╪═════════════════╪═══════╡
│ LIPON F Sanitary Formula Dish … ┆ null           ┆ 85.00           ┆ true  │
│ FINELINE Liquid Laundry Deterg… ┆ null           ┆ 89.00           ┆ true  │
│ PAO Win Wash Liquid Laundry De… ┆ null           ┆ 99.00           ┆ true  │
│ PAO Super White Laundry Deterg… ┆ null           ┆ 112.00          ┆ true  │
│ LIPON F Dishwashing Liquid Hyg… ┆ 84.00          ┆ 69.00           ┆ true  │
│ …                               ┆ …              ┆ …               ┆ …     │
│ HYGIENE Expert Care Concentrat… ┆ 60.00          ┆ 47.00           ┆ true  │
│ FINELIN

In [22]:
df_watchlist_final= df_watchlist_final.select([
    pl.col("name"),
    # strict=False จะเปลี่ยนทุกอย่างที่แปลงเป็น Float ไม่ได้ให้เป็น Null
    pl.col("promotion_price").cast(pl.Float64, strict=False),
    pl.col("original_price").cast(pl.Float64, strict=False),
    pl.col("condition")
])
df_prep_big_c_watchlist = re_evaluate_price(df_watchlist_final)
df_trans_big_c_watchlist = parse_product_names(df_prep_big_c_watchlist, "BigC")
df_trans_big_c_watchlist.write_excel(f"big_c_watchlist_{today_date}.xlsx")

In [23]:
search_results_df.write_excel(f"search_result_big_c_{today_date}.xlsx")

In [24]:
import os
import zipfile
from google.colab import files
import datetime

# Get the current date for the filename
today_date = datetime.datetime.now().strftime("%Y-%m-%d")

# Get all files in the current directory, excluding common Colab configuration files
files_to_zip = [f for f in os.listdir('.') if os.path.isfile(f) and not f.startswith('.') and not f.startswith('sample_data')]

print("Iterating through files in the current directory:")
for file in files_to_zip:
    print(f"- {file}")

zip_filename = f'bigc_{today_date}.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file)

print(f"\n'{zip_filename}' created successfully. Downloading now...")
files.download(zip_filename)

Iterating through files in the current directory:
- search_result_big_c_2026-08-07.xlsx
- big_c_watchlist_2026-08-07.xlsx
- big_c_result_2026-08-07.xlsx

'bigc_2026-08-07.zip' created successfully. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>